# ArNet2 - Preprocessing and Model Prediction

This notebook demonstrates the workflow for:
1. **Preprocessing R-peaks annotation data** from the **SHDB-AF** dataset for prediction.
2. **Running the trained ArNet2 model** for **AF prediction** using the preprocessed data.

---

### Dataset:
- **SHDB-AF**: The dataset used in this example is the [SHDB-AF dataset](https://physionet.org/content/shdb-af/1.0.1/), which contains ECG signals annotated with peak information and AF-related labels.

### Dataset Details:
- **ECG signals**: The data consists of ECG recordings, where each sample is labeled with corresponding **R-peaks annotations**.
- **R-peak annotations**: The location of R-peaks in the ECG signal.
- **AF labels per peak**: Each R-peak is annotated with a label indicating whether it is associated with **AF** or not.
- **Overall patient label**: The dataset includes a **global label** for each patient indicating the overall AF status (e.g., **PAF**: Paroxysmal AF, **Per**: Persistent AF, **Non-AF**).

This notebook will help demonstrate how to prepare the data for prediction and how to use the trained **ArNet2 model** to make predictions for **AF detection**.


### 1. Import Libraries

In [1]:
import os
import numpy as np
import pandas as pd
import wfdb
import requests
import pickle
import subprocess

### 2. Setting Up the Paths
We will first set up paths for data storage and where to save the processed data:

In [2]:
# Setting up the paths
data_path = '.././physionet_data'  # Where you'll download the PhysioNet dataset
output_path = '.././data'  # Where to save the processed data
os.makedirs(data_path, exist_ok=True)
os.makedirs(output_path, exist_ok=True)

### 3. Download the Data
Next, we will download the required ECG signal data and annotation files for SHDB-AF:

In [3]:
def download_file(url, filename):
    """
    Downloads a file from a specified URL and saves it to the given filename.

    :param url: URL to the file to be downloaded.
    :param filename: Path where the downloaded file will be saved.
    """
    response = requests.get(url)
    with open(filename, 'wb') as f:
        f.write(response.content)

# Example download for a single record
record_name = '001'
filepath = f'{data_path}/{record_name}'
url = f'https://physionet.org/files/shdb-af/1.0.1/{record_name}'

# Download annotation files and additional data
download_file(f'{url}.atr', f'{filepath}.atr')
download_file(f'{url}.qrs', f'{filepath}.qrs')
download_file(f'https://physionet.org/files/shdb-af/1.0.1/AdditionalData.csv', f'{data_path}/AdditionalData.csv')


### 4. Preprocess annotation Data

We will preprocess the R-peaks annotation data

In [4]:
# Load annotations (e.g., R-peaks) and patient data
annotation = wfdb.rdann(filepath, 'atr')
additionaldata = pd.read_csv(f'{data_path}/AdditionalData.csv')

# Calculate RR intervals
r_peaks = annotation.sample
rr_intervals = np.diff(r_peaks).astype(np.float32)  # Calculate RR intervals in samples

# Create a DataFrame for easier processing later
rr_time = r_peaks[1:] / annotation.fs  # Convert sample indices to time (seconds)
rr_data = rr_intervals / annotation.fs  # RR intervals in seconds

# Create DataFrame for prediction data
ecg_df = pd.DataFrame({
    'rr_data': rr_data,
    'rr_time': rr_time
})

window_size = 60  # 60 beats per window


### 5. Save the Preprocessed Data for Prediction
We save the processed R-peak annotation data as a CSV file, which will be used for prediction:

In [5]:
# Save the DataFrame as CSV
predict_file_path = os.path.join(output_path, f'{record_name}_df_test.csv')
ecg_df.to_csv(predict_file_path, index=False)

print(f"ECG data processing complete. File saved as {record_name}_df_test.csv")

ECG data processing complete. File saved as 001_df_test.csv


### 6. Run Prediction with the Trained Model
Finally, we use the trained ArNet2 model to make predictions based on the preprocessed data:

In [6]:
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
config_file_path = os.path.join(project_root, 'config', 'config.yml')
import os
import yaml

# Assuming config.yml is in the ./config/ directory of your project
config_file_path = os.path.join(project_root, 'config', 'config.yml')

# Load the config file (YAML format assumed)
with open(config_file_path, 'r') as file:
    config = yaml.safe_load(file)

# Convert relative paths to absolute paths based on project_root
config['path']['arnet2'] = os.path.abspath(os.path.join(project_root, config['path']['arnet2']))
config['path']['resnet'] = os.path.abspath(os.path.join(project_root, config['path']['resnet']))

# Save the updated config back to the file
updated_config_file_path = os.path.join(project_root, 'config', 'config_w_abs_path.yml')

with open(updated_config_file_path, 'w') as file:
    yaml.dump(config, file, default_flow_style=False)

subprocess.run(['python', '../run_ArNet2.py', '--mode', 'predict', '--input_file', predict_file_path, '--output_name', f'{record_name}_pred_df', '--save_output_path', '../results/', '--config', updated_config_file_path])

2025-09-03 00:11:22.427023: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-03 00:11:22.459545: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2025-09-03 00:11:23.328174: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


2025-09-03 00:11:25.585822: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1635] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 34067 MB memory:  -> device: 0, name: NVIDIA L40, pci bus id: 0000:02:00.0, compute capability: 8.9


2025-09-03 00:11:26.408076: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:424] Loaded cuDNN version 8903


2025-09-03 00:11:26.670141: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:637] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


Predicting...
3/3 [==============================] - 1s 61ms/step


3/3 [==============================] - 0s 3ms/step
Results saved to ../results/001_pred_df.csv


2025-09-03 00:11:27.108476: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype int32
	 [[{{node Placeholder/_0}}]]


CompletedProcess(args=['python', '../run_ArNet2.py', '--mode', 'predict', '--input_file', '.././data/001_df_test.csv', '--output_name', '001_pred_df', '--save_output_path', '../results/', '--config', '/home/shanybiton/repos/Shany_Repo/plug-and-play/config/config_w_abs_path.yml'], returncode=0)

### 8. Run Model Evaluation


This section evaluates the performance of the **ArNet2** model on the prediction results using various performance metrics such as **accuracy**, **F1-score**, **sensitivity**, **specificity**, **AUROC**, and **AUPRC**.

We will compare the model's predicted labels and the true labels to calculate these metrics.

In [7]:
def pad_rhythm(rhythm, missing=None):
    """
    Helper function which receives the changes in the cardiac rhythm labels and pads the whole vector.
        Example:
            in = ['AFIB', '', '', '', '', '', 'N', '', '', '', 'SBR', '']
            out = ['AFIB', 'AFIB', 'AFIB', 'AFIB', 'AFIB', 'AFIB', 'N', 'N', 'N', 'N', 'SBR', 'SBR']
        This function in used to parse the '.bea' files summarizing the beats detected in the UVAF database.
    :param rhythm: The input vector representing the changes in the cardiac rhythm (list of strings or labels).
    :param missing: The different strings or labels (list) which characterize a missing rhythm. (If None, considering only '' as a missing rhythm)
    :returns rhythm: The padded vector of rhythms.
    """
    cond = np.ones(len(rhythm), dtype=bool)
    if missing != None:
        for char in missing:
            cond = np.logical_and(cond, rhythm != char)
    else:
        cond = rhythm != missing
    not_none = np.where(cond)[0]
    if len(not_none) == 0:
        rhythm = np.array(['(N'] * len(rhythm))
    else:
        not_none = np.append(not_none, len(rhythm))
        diffs = np.diff(not_none)
        sing_rhy = rhythm[not_none[:-1]]
        rhythm[not_none[0]:] = np.repeat(sing_rhy, diffs)
        rhythm[0:not_none[0]] = sing_rhy[0]
    return rhythm


def calc_y(rhythm, window_size):
    """
    Generates window labels based on the rhythm sequence.

    :param rhythm: The sequence of rhythm labels (AF or non-AF).
    :param window_size: The size of each window in beats.
    :returns: Binary array with labels (1 for AF, 0 for non-AF).
    """
    rhythms = pad_rhythm(np.array(rhythm), missing=['', 'None'])
    rlab = rhythms[:(len(rhythms) // window_size) * window_size].reshape(-1, window_size)
    counts = np.sum((rlab == '(AFIB'), axis=1)
    return counts >= window_size // 2  # If half or more are AF, label as AF

y = calc_y(annotation.aux_note, window_size)

In [8]:
import utils.metrics as metrics
pred_df = pd.read_csv(f'../results/{record_name}_pred_df.csv')
accuracy, fbeta, sensitivity, specificity, PPV, NPV, AUROC, AUCPR = metrics.model_metrics(pred_df.proba, y, pred_df.pred, print_metrics=True)

Accuracy: 0.9995826377295493
F1-Score: 0.9997064866451424
Sensitivity: 1.0
Specificity: 0.9985569985569985
PPV: 0.9994131455399061
NPV: 1.0
AUROC: 0.9999728854690686
AUCPR: 0.9999888679302092
[[ 692    1]
 [   0 1703]]
